In [ ]:
"""
exp05 Step 01: Volatility Signal Check And Exploration (EXPLORATION ON 2005-2014 ONLY; PROTOCOL.md §4, §9)

1. SIGNAL CHECK (pre-registered gate): does the current daily_volatility (previous 20 daily returns) predict the realized
   volatility of the next 20 sessions? Spearman correlation on the decisions of 2005-01-03 -> 2014-12-31 with a circular
   block bootstrap of 6-month blocks. PASS if the 95% lower bound > 0. Expected to pass easily; if it does not, the
   pipeline is checked and step 02 is not run.
2. DESCRIPTIVE (the "leverage effect" question): forward 20-session log return by quintile of current volatility.
   Scaling exposure down can beat buy-and-hold only if the high-volatility bins have NEGATIVE forward returns.
3. EXPLORATION: the 4 candidates as one continuous path over 2005-2014 (targets fixed per calendar year from the earlier
   values; no target before 250 values exist, so w = 1 in 2005-2006), against buy-and-hold.

The data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of the walk-forward period is read.
Caveat: 2008 is known to everyone (hindsight-prone); the result does not change the grid or the selection (pre-registered).
Outputs: store04_experiments/exp05_vol_scaled_exposure/step01_exploration_data/. Trials: 1 signal_check + 4 exploration.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp05_vol_scaled_exposure import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE BUY AND HOLD BENCHMARK AND THE CONTINUOUS REPLAY SUMMARY
from so.core.reentry_simulation import get_buy_and_hold_result_dict
from so.core.continuous_replay import get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp05_vol_scaled_exposure.exposure import get_rule_candidate_list, get_weight_builder_func, simulate_weight_path_dict, \
                                                           get_volatility_signal_check_dict, get_volatility_bin_return_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THESE STAGES (A RERUN ADDS TRIALS)
previous_signal_count = print_previous_trial_warning(exp_config, "signal_check")
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

In [ ]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DEFINE ONE PERIOD PER CALENDAR YEAR (TARGETS FIXED PER YEAR; YEARS-WON COUNT)
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tWindow:\t{exploration_start_str} -> {exploration_end_str}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")

In [ ]:
"""
1. Signal check: current volatility vs the realized volatility of the next 20 sessions (PASS if the 95% lower bound > 0).
"""
# CALL FUNCTION TO RUN THE SIGNAL CHECK
signal_check_dict = get_volatility_signal_check_dict(daily_pdf, exploration_start_str, exploration_end_str)
print(f"Decisions:\t{signal_check_dict['row_count']:,}\tMonths:\t{signal_check_dict['month_count']}")
print(f"Spearman correlation:\t{signal_check_dict['spearman']:.3f}\t95% interval ({exp_config.SIGNAL_CHECK_BLOCK_MONTH_COUNT}-month blocks) [{signal_check_dict['ci_low']:.3f}, {signal_check_dict['ci_high']:.3f}]")
print(f"Log-log regression:\tslope {signal_check_dict['log_slope']:.3f}\tR^2 {signal_check_dict['log_r2']:.3f}")
print(f"\nSIGNAL CHECK PASSED:\t{signal_check_dict['passed']}")
# LOG THE SIGNAL CHECK (ONE TRIAL)
signal_trial_dict = log_trial_dict(exp_config, "signal_check", {"predictor": exp_config.VOLATILITY_COL_STR, "target": f"realized volatility next {exp_config.SIGNAL_CHECK_HORIZON_SESSIONS} sessions",
                                                                 "window": f"{exploration_start_str}->{exploration_end_str}"},
                                   signal_check_dict, False, note_str_in="exp05 step01 volatility signal check (gate of step 02)")
print(f"Signal check logged:\t{signal_trial_dict['trial_id']}")

In [ ]:
"""
2. Descriptive: forward 20-session log return (15:59 fill to 15:59 fill) by quintile of current volatility.
"""
# CALL FUNCTION TO BUILD THE BIN TABLE
bin_pdf = get_volatility_bin_return_pdf(daily_pdf, exploration_start_str, exploration_end_str)
all_mean = float(np.average(bin_pdf["mean_forward_log_return"], weights=bin_pdf["row_count"]))
print(f"Mean forward 20-session log return over all decisions:\t{all_mean:.4%}")
# PREVIEW DATAFRAME
bin_pdf.round(5)

In [ ]:
"""
3. Exploration: the 4 candidates, each one continuous path, against buy-and-hold (and the constant exposure at its mean weight).
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE WEIGHT BUILDER (TARGETS FIXED PER YEAR)
weight_builder_func = get_weight_builder_func(daily_pdf, year_period_pdf)
# LIST TO HOLD THE RESULTS
row_dict_list = []
# ITERATE OVER THE CANDIDATES
for candidate_idx, candidate_dict in enumerate(get_rule_candidate_list()):
    # BUILD AND REPLAY THE CANDIDATE
    weight_arr = weight_builder_func(candidate_dict)
    simulation_dict = simulate_weight_path_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, weight_arr, exp_config.UNTOUCHED_START_DATE_STR)
    # REPLAY THE CONSTANT EXPOSURE AT THE SAME MEAN WEIGHT
    constant_arr = np.full(len(weight_arr), simulation_dict["metric_dict"]["mean_weight"])
    constant_dict = simulate_weight_path_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, constant_arr, exp_config.UNTOUCHED_START_DATE_STR)
    # SUMMARIZE THE PATH (YEARS AS PERIODS)
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    # STORE THE ROW
    row_dict_list.append({"candidate_idx": candidate_idx, **candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "constant_exposure_total_return": constant_dict["metric_dict"]["total_return"],
                          "mean_weight": simulation_dict["metric_dict"]["mean_weight"], "rebalance_count": simulation_dict["metric_dict"]["rebalance_count"],
                          "annualized_return": summary_dict["strategy_annualized_return"], "sharpe_ratio": summary_dict["strategy_sharpe_ratio"],
                          "constant_exposure_sharpe_ratio": constant_dict["metric_dict"]["sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"],
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS
exploration_pdf = pd.DataFrame(row_dict_list)
# PREVIEW DATAFRAME
exploration_pdf.round(4)

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(pd.DataFrame([signal_check_dict]), f"{output_path_str}signal_check_data.csv", "W")
write_csv_file_to_path(bin_pdf, f"{output_path_str}volatility_bin_return_data.csv", "W")
write_csv_file_to_path(exploration_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")

In [ ]:
# LOG THE 4 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["floor", "target_quantile", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["candidate_idx", "floor", "target_quantile"]], False,
                                      note_str_in="exp05 step01 exploration (2005-2014 continuous fractional replay, context only)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)